# StockSage AI — LSTM Training
Run on **Kaggle Notebooks** (free T4 GPU).  
Trains a bidirectional LSTM for sequence-based price prediction.

In [ ]:
!pip install yfinance pandas-ta --quiet
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import yfinance as yf
import pandas_ta as ta

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

SEQ_LEN = 30       # 30 days lookback
HORIZON = 7        # predict 7-day return
BATCH_SIZE = 64
EPOCHS = 50
HIDDEN = 128
NUM_LAYERS = 2
DROPOUT = 0.3

In [ ]:
SYMBOLS = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'BHARTIARTL.NS','KOTAKBANK.NS','LT.NS','SBIN.NS','BAJFINANCE.NS',
]

FEATURE_COLS = ['close_norm', 'volume_norm', 'RSI_14', 'MACD_12_26_9', 'BBP_20_2.0']

def prepare_data(symbol: str):
    ticker = yf.Ticker(symbol)
    df = ticker.history(period='5y', interval='1d')
    if df.empty or len(df) < 100:
        return None, None
    
    df = df.reset_index()
    df.columns = [c.lower() for c in df.columns]
    
    # Normalise
    df['close_norm'] = df['close'] / df['close'].shift(1) - 1
    df['volume_norm'] = (df['volume'] - df['volume'].mean()) / (df['volume'].std() + 1e-8)
    
    df.ta.rsi(length=14, append=True)
    df.ta.macd(append=True)
    df.ta.bbands(length=20, append=True)
    df = df.dropna()
    
    # Normalise indicator columns
    for col in ['RSI_14', 'MACD_12_26_9', 'BBP_20_2.0']:
        std = df[col].std()
        if std > 0:
            df[col] = (df[col] - df[col].mean()) / std
    
    # Target: binary — 1 if 7-day return > 0, else 0
    df['target'] = (df['close'].shift(-HORIZON) > df['close']).astype(int)
    df = df.dropna()
    
    X, y = [], []
    for i in range(SEQ_LEN, len(df)):
        X.append(df[FEATURE_COLS].iloc[i-SEQ_LEN:i].values)
        y.append(df['target'].iloc[i])
    
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

In [ ]:
all_X, all_y = [], []
for sym in SYMBOLS:
    X, y = prepare_data(sym)
    if X is not None:
        all_X.append(X)
        all_y.append(y)
        print(f'{sym}: {len(X)} sequences')

X_all = np.concatenate(all_X)
y_all = np.concatenate(all_y)
print(f'Total: {X_all.shape}, target distribution: {np.bincount(y_all)}')

In [ ]:
# Train/val split (temporal — no shuffle)
split = int(0.85 * len(X_all))
X_train = torch.tensor(X_all[:split])
y_train = torch.tensor(y_all[:split])
X_val = torch.tensor(X_all[split:])
y_val = torch.tensor(y_all[split:])

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, n_features, hidden, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, n_layers, batch_first=True,
                            dropout=dropout, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

model = StockLSTM(len(FEATURE_COLS), HIDDEN, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
best_val_acc = 0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_correct = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds = model(xb).argmax(1)
            val_correct += (preds == yb).sum().item()
    val_acc = val_correct / len(val_ds)
    scheduler.step()
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'lstm_weights.pt')
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}: loss={train_loss/len(train_dl):.4f}, val_acc={val_acc:.3f}')

print(f'\nBest val accuracy: {best_val_acc:.3f}')
print('Saved: lstm_weights.pt')

### Upload to backend
Download `lstm_weights.pt` from Kaggle and place at `backend/app/models/lstm_weights.pt`

To use it in inference, load it as:
```python
model = StockLSTM(n_features=5, hidden=128, n_layers=2, dropout=0.3)
model.load_state_dict(torch.load('lstm_weights.pt', map_location='cpu'))
model.eval()
```